<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Market%20Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Jason Market Solver Test v1

Formål:
- Leser Market_xG_All_Fixtures_v1 fra Jason Development.
- Leser evt. Market_Team_Strength_v2_1 for sammenligning.
- Prøver å rekonstruere markedets kamp-xG ved å løse implisitte lagstyrker:

    xG(team vs opponent) = mu * attack(team) * def_weakness(opponent) * home_factor

  der:
    attack > 1        = sterkere angrep enn ligaen
    def_weakness > 1  = svakere forsvar enn ligaen, slipper til mer xG
    def_strength = 1 / def_weakness
    home_factor = gamma for hjemmelag, 1/gamma for bortelag

- Skriver resultatene til egne testark:
    Market_Solver_Summary_v1
    Market_Solver_Ratings_v1
    Market_Solver_Check_GW1_v1

Viktig:
- Dette er et isolert testscript. Det endrer ikke TO, Market_xP eller Jason-logikken.
- Med bare én GW er problemet underbestemt. Derfor bruker scriptet regularisering
  mot nøytrale ratings (= 1.0). Det gir en stabil første løsning.
"""

import math
import time
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

from google.colab import auth
import gspread
from google.auth import default

# -----------------------
# KONFIGURASJON
# -----------------------

SPREADSHEET_NAME = "Jason Development"
MARKET_XG_SHEET = "Market_xG_All_Fixtures_v1"
TEAM_STRENGTH_SHEET = "Market_Team_Strength_v2_1"

SUMMARY_OUT = "Market_Solver_Summary_v1"
RATINGS_OUT = "Market_Solver_Ratings_v1"
CHECK_OUT = "Market_Solver_Check_GW1_v1"

# Regularisering: høyere = mer konservativ mot 1.0.
# 0.15-0.40 er fornuftig start når vi bare har én GW.
REG_STRENGTH = 0.25

# Identifiserbarhet / anker: holder gjennomsnittlig log attack og def weakness nær 0.
ANCHOR_STRENGTH = 2.0

# Hvis True estimeres gamma. Hvis False brukes FIXED_GAMMA.
ESTIMATE_GAMMA = True
FIXED_GAMMA = 1.10

# Rimelige bounds for én-GW-løsningen.
ATTACK_MIN, ATTACK_MAX = 0.35, 2.80
DEF_WEAK_MIN, DEF_WEAK_MAX = 0.35, 2.80
GAMMA_MIN, GAMMA_MAX = 0.90, 1.30

# -----------------------
# Google Sheets helpers
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def parse_float(v, default=np.nan):
    if v is None:
        return default
    s = str(v).strip()
    if s == "":
        return default
    s = s.replace(" ", "")
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    elif "," in s and "." in s:
        s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return default


def read_sheet(sheet_name):
    ws = sh.worksheet(sheet_name)
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()
    header = values[0]
    rows = values[1:]
    return pd.DataFrame(rows, columns=header)


def update_worksheet(sheet_name, dataframe, min_rows=100, min_cols=30, sleep_seconds=1.5):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows=str(min_rows), cols=str(min_cols))

    ws.resize(
        rows=max(min_rows, len(df_clean) + 20),
        cols=max(min_cols, len(df_clean.columns) + 5),
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(sleep_seconds)


# -----------------------
# Dataklargjøring
# -----------------------

xg_df = read_sheet(MARKET_XG_SHEET)
required = ["Team", "Opponent", "H/A", "Team xG", "Opp xG", "CS%"]
missing = [c for c in required if c not in xg_df.columns]
if missing:
    raise ValueError(f"Mangler kolonner i {MARKET_XG_SHEET}: {missing}")

# Rens / parse
for col in ["Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG"]:
    if col in xg_df.columns:
        xg_df[col] = xg_df[col].apply(parse_float)

xg_df["Team"] = xg_df["Team"].astype(str).str.strip()
xg_df["Opponent"] = xg_df["Opponent"].astype(str).str.strip()
xg_df["H/A"] = xg_df["H/A"].astype(str).str.strip().str.upper()
xg_df = xg_df.dropna(subset=["Team xG"]).copy()

if xg_df.empty:
    raise ValueError(f"Ingen gyldige xG-rader i {MARKET_XG_SHEET}")

teams = sorted(set(xg_df["Team"]).union(set(xg_df["Opponent"])))
team_to_i = {t: i for i, t in enumerate(teams)}
n = len(teams)

mu = float(xg_df["Team xG"].mean())
print(f"Lag: {n}")
print(f"Rader: {len(xg_df)}")
print(f"mu / league avg xG: {mu:.4f}")

# -----------------------
# Solver-modell
# -----------------------

# Parametervektor:
# [log_attack_0..n-1, log_defweak_0..n-1, optional log_gamma]

log_attack_lb = math.log(ATTACK_MIN)
log_attack_ub = math.log(ATTACK_MAX)
log_def_lb = math.log(DEF_WEAK_MIN)
log_def_ub = math.log(DEF_WEAK_MAX)
log_gamma_lb = math.log(GAMMA_MIN)
log_gamma_ub = math.log(GAMMA_MAX)

if ESTIMATE_GAMMA:
    x0 = np.zeros(2 * n + 1)
    x0[-1] = math.log(FIXED_GAMMA)
    lower = np.array([log_attack_lb] * n + [log_def_lb] * n + [log_gamma_lb])
    upper = np.array([log_attack_ub] * n + [log_def_ub] * n + [log_gamma_ub])
else:
    x0 = np.zeros(2 * n)
    lower = np.array([log_attack_lb] * n + [log_def_lb] * n)
    upper = np.array([log_attack_ub] * n + [log_def_ub] * n)


def unpack(params):
    log_att = params[:n]
    log_defweak = params[n:2*n]
    if ESTIMATE_GAMMA:
        gamma = math.exp(params[-1])
    else:
        gamma = FIXED_GAMMA
    return log_att, log_defweak, gamma


def predict_log_xg_for_row(row, log_att, log_defweak, gamma):
    team = row["Team"]
    opp = row["Opponent"]
    ha = row["H/A"]
    t = team_to_i[team]
    o = team_to_i[opp]

    if ha == "H":
        hfac = gamma
    elif ha == "A":
        hfac = 1.0 / gamma
    else:
        hfac = 1.0

    return math.log(mu) + log_att[t] + log_defweak[o] + math.log(hfac)


def residuals(params):
    log_att, log_defweak, gamma = unpack(params)
    res = []

    # Data residuals in log space: relative fit matters more than absolute xG level.
    for _, row in xg_df.iterrows():
        y = float(row["Team xG"])
        if y <= 0:
            continue
        pred_log = predict_log_xg_for_row(row, log_att, log_defweak, gamma)
        res.append(pred_log - math.log(y))

    # Regularisering mot nøytral rating 1.0.
    if REG_STRENGTH > 0:
        res.extend((REG_STRENGTH * log_att).tolist())
        res.extend((REG_STRENGTH * log_defweak).tolist())

    # Anker for identifiserbarhet: gjennomsnittlig log-rating nær 0.
    if ANCHOR_STRENGTH > 0:
        res.append(ANCHOR_STRENGTH * np.mean(log_att))
        res.append(ANCHOR_STRENGTH * np.mean(log_defweak))

    return np.array(res, dtype=float)


result = least_squares(
    residuals,
    x0,
    bounds=(lower, upper),
    max_nfev=20000,
    xtol=1e-10,
    ftol=1e-10,
    gtol=1e-10,
)

log_att, log_defweak, gamma = unpack(result.x)
attack = np.exp(log_att)
def_weak = np.exp(log_defweak)
def_strength = 1.0 / def_weak

# -----------------------
# GW1 check / fasit mot modell
# -----------------------

check_rows = []
for _, row in xg_df.iterrows():
    pred_log = predict_log_xg_for_row(row, log_att, log_defweak, gamma)
    pred_xg = math.exp(pred_log)
    market_xg = float(row["Team xG"])
    market_opp_xg = parse_float(row.get("Opp xG", np.nan))
    market_cs = parse_float(row.get("CS%", np.nan))

    # Predikert CS for laget = exp(-predikert opp xG).
    # Finn motsatt rad i samme kamp hvis mulig: same team/opponent swapped.
    opp_rows = xg_df[(xg_df["Team"].eq(row["Opponent"])) & (xg_df["Opponent"].eq(row["Team"]))]
    pred_opp_xg = np.nan
    if not opp_rows.empty:
        opp_row = opp_rows.iloc[0]
        pred_opp_xg = math.exp(predict_log_xg_for_row(opp_row, log_att, log_defweak, gamma))
    else:
        # fallback: bruk markedets opp xG om match-pair ikke finnes
        pred_opp_xg = market_opp_xg

    pred_cs = math.exp(-pred_opp_xg) * 100 if not pd.isna(pred_opp_xg) else np.nan

    check_rows.append({
        "Kickoff": row.get("Kickoff", ""),
        "Team": row["Team"],
        "Opponent": row["Opponent"],
        "H/A": row["H/A"],
        "Market xG": market_xg,
        "Solver xG": pred_xg,
        "xG Diff": pred_xg - market_xg,
        "Abs xG Diff": abs(pred_xg - market_xg),
        "Market Opp xG": market_opp_xg,
        "Solver Opp xG": pred_opp_xg,
        "Market CS%": market_cs,
        "Solver CS%": pred_cs,
        "CS Diff": pred_cs - market_cs if not pd.isna(market_cs) and not pd.isna(pred_cs) else np.nan,
    })

check_df = pd.DataFrame(check_rows)

mae_xg = float(check_df["Abs xG Diff"].mean())
rmse_xg = float(np.sqrt(np.mean(np.square(check_df["xG Diff"]))))
mae_cs = float(check_df["CS Diff"].abs().mean()) if "CS Diff" in check_df else np.nan

# -----------------------
# Ratings-tabell + sammenligning med eksisterende Market Team Strength
# -----------------------

ratings_df = pd.DataFrame({
    "Team": teams,
    "Solver Attack": attack,
    "Solver Def Weakness": def_weak,
    "Solver Def Strength": def_strength,
})

# Nøytraliser små skala-avvik: gjennomsnitt rundt 1 er enklere å lese.
ratings_df["Solver Attack Index"] = ratings_df["Solver Attack"] / ratings_df["Solver Attack"].mean()
ratings_df["Solver DefWeak Index"] = ratings_df["Solver Def Weakness"] / ratings_df["Solver Def Weakness"].mean()
ratings_df["Solver DefStrength Index"] = ratings_df["Solver Def Strength"] / ratings_df["Solver Def Strength"].mean()

# Optional compare with current team strength sheet.
try:
    ts_df = read_sheet(TEAM_STRENGTH_SHEET)
    if not ts_df.empty and "Team" in ts_df.columns:
        ts_df["Team"] = ts_df["Team"].astype(str).str.strip()
        for col in ["Team Att Ratio", "Team Def Ratio", "Avg Team xG", "Avg Opp xG", "Team Att", "Team Def"]:
            if col in ts_df.columns:
                ts_df[col] = ts_df[col].apply(parse_float)
        keep_cols = [c for c in ["Team", "Team Att Ratio", "Team Def Ratio", "Avg Team xG", "Avg Opp xG", "Team Att", "Team Def"] if c in ts_df.columns]
        ratings_df = ratings_df.merge(ts_df[keep_cols], on="Team", how="left")
except Exception as e:
    print(f"Kunne ikke lese {TEAM_STRENGTH_SHEET} for sammenligning: {e}")

# Sorter etter solver attack først, deretter sterkest defence.
ratings_df = ratings_df.sort_values(["Solver Attack Index", "Solver DefStrength Index"], ascending=[False, False]).reset_index(drop=True)

# -----------------------
# Summary
# -----------------------

summary_rows = [
    {"Metric": "Input sheet", "Value": MARKET_XG_SHEET},
    {"Metric": "Rows used", "Value": len(xg_df)},
    {"Metric": "Teams", "Value": n},
    {"Metric": "mu / league avg xG", "Value": mu},
    {"Metric": "Estimate gamma", "Value": ESTIMATE_GAMMA},
    {"Metric": "Gamma", "Value": gamma},
    {"Metric": "Regularization", "Value": REG_STRENGTH},
    {"Metric": "Anchor strength", "Value": ANCHOR_STRENGTH},
    {"Metric": "xG MAE", "Value": mae_xg},
    {"Metric": "xG RMSE", "Value": rmse_xg},
    {"Metric": "CS MAE pct points", "Value": mae_cs},
    {"Metric": "Solver success", "Value": result.success},
    {"Metric": "Solver message", "Value": result.message},
]
summary_df = pd.DataFrame(summary_rows)

# Round numeric columns for sheet readability.
for df in [ratings_df, check_df, summary_df]:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].round(4)

# -----------------------
# Write sheets
# -----------------------

update_worksheet(SUMMARY_OUT, summary_df, min_rows=100, min_cols=10)
update_worksheet(RATINGS_OUT, ratings_df, min_rows=100, min_cols=30)
update_worksheet(CHECK_OUT, check_df, min_rows=200, min_cols=30)

print("")
print("FERDIG: Jason Market Solver Test v1")
print(f"Gamma: {gamma:.4f}")
print(f"xG MAE: {mae_xg:.4f}")
print(f"xG RMSE: {rmse_xg:.4f}")
print(f"CS MAE: {mae_cs:.2f} prosentpoeng")
print("")
print("Sjekk arkene:")
print(f"- {SUMMARY_OUT}")
print(f"- {RATINGS_OUT}")
print(f"- {CHECK_OUT}")

Lag: 20
Rader: 20
mu / league avg xG: 1.5430
Oppdaterer Market_Solver_Summary_v1...
Oppdaterer Market_Solver_Ratings_v1...
Oppdaterer Market_Solver_Check_GW1_v1...
